# Minimal L3C maps

Reuse the projection, extent and variable presets from `src/driftaware_sialt/visualization`. Coastlines and labels are light; the map, land, ocean and missing data have no fill. The dark background belongs only to the notebook preview, so PNG exports stay transparent.

Run with the project's Python environment. Change `test_file` or `variable` in the example below. Hemisphere is read from the NetCDF projection.

In [ ]:
from pathlib import Path
from io import BytesIO
import base64

import cartopy.feature as cfeature
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm
import numpy as np
import xarray as xr
from IPython.display import HTML, display

from driftaware_sialt.visualization.visualization import PLOT_PRESETS
from driftaware_sialt.visualization.visualization_tools import _projection_and_extent


In [ ]:
def plot_l3c(
    path,
    variable="sea_ice_thickness",
    *,
    time_index=0,
    figsize=(7, 7),
    vmin=None,
    vmax=None,
    cmap=None,
    coastline_color="#cbd5e1",
    coastline_width=0.45,
    show_date=True,
    show_colorbar=True,
):
    """Return (figure, axes, mesh) for a transparent, dark-friendly L3C map.

    Projection, extent, value scaling and default colour limits match the source
    visualisation presets. vmin/vmax use the preset's display units (e.g. cm for
    freeboard). No land/ocean fill or global Matplotlib style is applied.
    """
    if variable not in PLOT_PRESETS:
        raise ValueError(f"No source visualisation preset for {variable!r}")
    preset = PLOT_PRESETS[variable]
    with xr.open_dataset(path) as ds:
        field = ds[variable]
        if "time" in field.dims:
            field = field.isel(time=time_index)
        values = field.transpose("yc", "xc").load().values * preset.scale
        x, y = ds.xc.values.copy(), ds.yc.values.copy()
        grid_mapping = ds[field.attrs.get("grid_mapping", "crs")]
        origin = float(grid_mapping.attrs["latitude_of_projection_origin"])
        if origin not in (-90.0, 90.0):
            raise ValueError("Expected a polar Lambert azimuthal equal-area L3C grid")
        hemisphere = "nh" if origin > 0 else "sh"
        date = (
            ds.time.isel(time=time_index).dt.strftime("%Y-%m-%d").item()
            if "time" in ds.coords else ""
        )

    projection, extent = _projection_and_extent(hemisphere)
    low = preset.vmin if vmin is None else vmin
    high = preset.vmax if vmax is None else vmax
    if high <= low:
        raise ValueError("vmax must be greater than vmin")
    palette = (plt.get_cmap(cmap) if isinstance(cmap, str) else cmap or preset.cmap).copy()
    palette.set_bad((0, 0, 0, 0))
    norm = BoundaryNorm(np.linspace(low, high, preset.levels), palette.N)

    fig, ax = plt.subplots(
        figsize=figsize, subplot_kw={"projection": projection}, facecolor="none"
    )
    ax.set_facecolor("none")
    ax.set_extent(extent, crs=projection)
    mesh = ax.pcolormesh(
        x, y, np.ma.masked_invalid(values),
        transform=projection, shading="auto", cmap=palette, norm=norm,
        rasterized=True, zorder=1,
    )
    ax.coastlines(
        resolution="50m", color=coastline_color, linewidth=coastline_width, zorder=3
    )
    if hemisphere == "sh":
        shelves = cfeature.NaturalEarthFeature(
            "physical", "antarctic_ice_shelves_polys", "50m",
            facecolor="none", edgecolor=coastline_color,
        )
        ax.add_feature(shelves, linewidth=coastline_width, zorder=3)
    ax.set_axis_off()

    if show_date and date:
        ax.text(
            0.015, 0.985, date, transform=ax.transAxes,
            ha="left", va="top", color=coastline_color, fontsize=10,
        )
    if show_colorbar:
        cax = ax.inset_axes([0.1, -0.055, 0.8, 0.025])
        cax.set_facecolor("none")
        colorbar = fig.colorbar(mesh, cax=cax, orientation="horizontal")
        colorbar.set_ticks(np.linspace(low, high, 3))
        colorbar.set_label(preset.label, color=coastline_color, fontsize=10, labelpad=5)
        colorbar.ax.tick_params(colors=coastline_color, labelsize=9, length=0, pad=5)
        colorbar.outline.set_visible(False)

    fig.subplots_adjust(left=0, right=1, bottom=0.1, top=1)
    return fig, ax, mesh


In [ ]:
def show_dark(fig, background="#101418"):
    """Preview a transparent figure on a dark notebook-only background."""
    buffer = BytesIO()
    fig.savefig(
        buffer, format="png", transparent=True,
        dpi=160, bbox_inches="tight", pad_inches=0.12,
    )
    encoded = base64.b64encode(buffer.getvalue()).decode("ascii")
    display(HTML(
        f'<div style="background:{background};padding:16px;max-width:760px;'
        f'border-radius:8px;line-height:0;">'
        f'<img src="data:image/png;base64,{encoded}" alt="L3C polar map" '
        f'style="display:block;width:100%;height:auto;">'
        f'</div>'
    ))


In [ ]:
test_file = Path(
    "/Volumes/data1/projects/CCI_SeaIce/output_data/v2.0/l3c/"
    "cryosat2_sentinel3a_sentinel3b/nh/2018/12/"
    "ESACCI-SEAICE-L3C-DA-SITHICK-"
    "SIRAL_CRYOSAT2_SRAL_SENTINEL3A_SRAL_SENTINEL3B-"
    "NH_12KM_EASE2-20181201-fv2.0.nc"
)
variable = "sea_ice_thickness"

fig, ax, mesh = plot_l3c(test_file, variable=variable)
show_dark(fig)
plt.close(fig)


Export with `transparent=True` to preserve the clear background. The returned figure remains available after closing its notebook display.

In [ ]:
# Transparent export: uncomment to save beside this notebook.
# fig.savefig("sea_ice_thickness_20181201.png",
#             dpi=300, transparent=True, bbox_inches="tight", pad_inches=0.12)

# Other source presets include sea_ice_thickness_total_unc, snow_depth,
# sea_ice_concentration, time_offset_acquisition and divergence.
# To customise: plot_l3c(test_file, vmin=0, vmax=4, cmap="magma").


## SAR context: 1 December 2018 ±5 days

The DTU archive supplies daily 1 km Sentinel-1A/B **display mosaics** for all dates from 26 November to 6 December 2018. The cells below cache these images and their original VRT georeferencing, then build a **2 km EPSG:6931 mosaic** on the same extent as the thickness plot. Pixels from the closest daily image win; earlier dates win ties. Gaps remain transparent.

These JPEG brightness values are for visual context, **not calibrated backscatter**. Images are not corrected for sea-ice motion. The saved `source_day_offset` records the daily mosaic date relative to the target, not an exact scene acquisition time. The provider's source CRS uses a sphere: the reprojection reads its VRT rather than assuming EPSG:3413.

First execution downloads about 60 MB into `notebooks/data/sar/`; subsequent runs use the cache. The combined NetCDF is about 5 MB. Source: [DTU archive, 1 December 2018](http://seaice.dk/N/2018/12/01/). Credit: Copernicus Sentinel data, processed by DTU Space / seaice.dk.

In [ ]:
from driftaware_sialt.visualization.sar import build_sar_mosaic

notebook_dir = Path("notebooks") if Path("notebooks").is_dir() else Path(".")
sar_cache = notebook_dir / "data" / "sar"
with xr.open_dataset(test_file) as source:
    sar_date = source.time.isel(time=0).dt.strftime("%Y-%m-%d").item()

sar_days = 5
sar_resolution = 2000  # metres
sar_file = sar_cache / f"{sar_date.replace('-', '')}-pm{sar_days}d-{sar_resolution}m-sar.nc"
if sar_file.exists():
    sar = xr.load_dataset(sar_file, decode_timedelta=False)
else:
    sar = build_sar_mosaic(sar_date, sar_cache, days=sar_days, resolution=sar_resolution)
    sar_file.parent.mkdir(parents=True, exist_ok=True)
    temporary = sar_file.with_suffix(".nc.part")
    sar.to_netcdf(temporary, engine="netcdf4")
    temporary.replace(sar_file)

print(f"SAR window: {sar.attrs['window_start']} to {sar.attrs['window_end']}")
print(f"Resolution: {sar.attrs['resolution_m'] / 1000:g} km; EPSG:6931")
print(f"Cached mosaic: {sar_file.resolve()}")


In [ ]:
def add_sar_background(ax, sar, *, alpha=1.0, vmin=20, vmax=220):
    """Add the georeferenced SAR display layer beneath an existing NH map."""
    from pyproj import CRS

    if CRS.from_wkt(sar.crs.attrs["spatial_ref"]).to_epsg() != 6931:
        raise ValueError("Expected the EPSG:6931 Arctic SAR mosaic")
    projection, _ = _projection_and_extent("nh")
    step = float(sar.attrs["resolution_m"])
    extent = [
        float(sar.xc[0]) - step / 2, float(sar.xc[-1]) + step / 2,
        float(sar.yc[-1]) - step / 2, float(sar.yc[0]) + step / 2,
    ]
    values = sar.sar_brightness.values
    clear = np.ma.masked_where(~np.isfinite(values) | (values <= 15), values)
    palette = plt.get_cmap("gray").copy()
    palette.set_bad((0, 0, 0, 0))
    return ax.imshow(
        clear, extent=extent, origin="upper", transform=projection,
        interpolation="nearest", cmap=palette, vmin=vmin, vmax=vmax,
        alpha=alpha, zorder=0.5,
    )


def plot_sar(sar, *, figsize=(7, 7)):
    """Plot SAR context with the same map extent and transparent styling."""
    projection, extent = _projection_and_extent("nh")
    fig, ax = plt.subplots(
        figsize=figsize, subplot_kw={"projection": projection}, facecolor="none"
    )
    ax.set_facecolor("none")
    ax.set_extent(extent, crs=projection)
    image = add_sar_background(ax, sar)
    ax.coastlines(resolution="50m", color="#cbd5e1", linewidth=0.45, zorder=3)
    ax.set_axis_off()
    ax.text(
        0.015, 0.985,
        f"{sar.attrs['window_start']} – {sar.attrs['window_end']}",
        transform=ax.transAxes, ha="left", va="top",
        color="#cbd5e1", fontsize=10,
    )
    ax.text(
        0.5, -0.035,
        f"Sentinel-1A/B · DTU Space · {sar.attrs['resolution_m'] / 1000:g} km · display brightness",
        transform=ax.transAxes, ha="center", va="top",
        color="#cbd5e1", fontsize=9,
    )
    fig.subplots_adjust(left=0, right=1, bottom=0.07, top=1)
    return fig, ax, image


In [ ]:
sar_fig, sar_ax, sar_image = plot_sar(sar)
show_dark(sar_fig)
plt.close(sar_fig)

# Transparent export:
# sar_fig.savefig("sar_20181201_pm5days.png",
#                 dpi=300, transparent=True, bbox_inches="tight", pad_inches=0.12)


In [ ]:
# Optional: place SAR beneath the coloured thickness map.
# overlay_fig, overlay_ax, _ = plot_l3c(test_file, variable=variable)
# add_sar_background(overlay_ax, sar, alpha=0.7)
# show_dark(overlay_fig)
# plt.close(overlay_fig)

# Pixel ages in days are available as sar.source_day_offset.
# Locations without valid SAR observations remain transparent.
